# SIC8 — 08 | Continuous End-to-End Streaming Pipeline (Bronze → Silver → CDC Enriched)

**24/7 Production Streaming Engine.**
This notebook runs the complete end-to-end streaming pipeline as concurrent Structured Streaming queries on a single compute cluster:
1. **Stream 1 (Bronze AutoLoader):** Ingests new S3 `.json.gz` files from `s3://sic-pipeline-storage/bronze/transactions/` into `streaming_raw_envelopes_v3`.
2. **Stream 2 (Silver Contract & CDC Enrichment):** Consumes raw Delta envelopes, parses strict contracts, dedupes events, isolates ground truth, enriches with CDC users and cards, and writes clean transactions to `streaming_transactions_enriched_v2`.

Runs continuously with `processingTime='10 seconds'` and awaits termination.

In [ ]:
# 1. Environment & Configuration
CATALOG = "workspace"
SCHEMA = "sic8_streaming"
SOURCE_S3 = "s3://sic-pipeline-storage/bronze/transactions/"

STATE_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/streaming_state"
RAW_CHECKPOINT = f"{STATE_VOLUME}/autoloader_tx_raw_v3/checkpoint"
RAW_SCHEMA_PATH = f"{STATE_VOLUME}/autoloader_tx_raw_v3/schema"
SILVER_CHECKPOINT = f"{STATE_VOLUME}/silver_transactions_v2/checkpoint"

RAW_TABLE = f"{CATALOG}.{SCHEMA}.streaming_raw_envelopes_v3"
TX_TABLE = f"{CATALOG}.{SCHEMA}.streaming_transactions_v2"
ENRICHED_TX_TABLE = f"{CATALOG}.{SCHEMA}.streaming_transactions_enriched_v2"
TRUTH_TABLE = f"{CATALOG}.{SCHEMA}.streaming_synthetic_truth_v2"
QUAR_TABLE = f"{CATALOG}.{SCHEMA}.streaming_quarantine_v2"
REG_TABLE = f"{CATALOG}.{SCHEMA}.streaming_event_registry_v2"
PREDS_TABLE = f"{CATALOG}.{SCHEMA}.streaming_model_predictions_v1"
DRIFT_TABLE = f"{CATALOG}.{SCHEMA}.streaming_model_drift_metrics_v1"
DIM_USERS = f"{CATALOG}.sic8_silver.historical_users"
DIM_CARDS = f"{CATALOG}.sic8_silver.historical_cards"

TRIGGER_INTERVAL = "10 seconds"
MAX_FILES_PER_TRIGGER = 500

from pyspark.sql import functions as F, types as T, Window
from datetime import datetime, timezone
spark.conf.set("spark.sql.session.timeZone", "UTC")

print("PIPELINE CONFIGURATION READY")
print("SOURCE S3:", SOURCE_S3)
print("RAW TABLE:", RAW_TABLE)
print("ENRICHED TABLE:", ENRICHED_TX_TABLE)


## 2. Ensure Destinations & Schemas Exist

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`streaming_state`")

# Ensure Raw Delta Table
spark.sql(f"""CREATE TABLE IF NOT EXISTS {RAW_TABLE} (
    raw_json_line STRING, source_file STRING, source_modified_utc TIMESTAMP,
    source_size_bytes BIGINT, event_id STRING, event_type STRING,
    schema_version STRING, kafka_offset BIGINT, kafka_key STRING,
    ingested_at_utc TIMESTAMP
) USING DELTA""")

# Ensure Silver Analytical Tables
spark.sql(f"""CREATE TABLE IF NOT EXISTS {TX_TABLE} (
    event_id STRING, event_type STRING, schema_version STRING,
    transaction_id STRING, source_transaction_id STRING, transaction_id_origin STRING,
    user_id BIGINT, user_id_origin STRING, card_index BIGINT, scoped_card_key STRING,
    amount DECIMAL(18,2), currency STRING, currency_origin STRING,
    merchant_name STRING, merchant_city STRING, merchant_state STRING,
    merchant_latitude DOUBLE, merchant_longitude DOUBLE, zip STRING, mcc INT,
    use_chip STRING, transaction_entry_method STRING, errors STRING,
    source_system STRING, event_time_utc TIMESTAMP, source_event_time_local TIMESTAMP_NTZ,
    event_payload_hash STRING, source_file STRING, source_modified_utc TIMESTAMP,
    kafka_partition INT, kafka_offset BIGINT, kafka_key STRING, ingested_at_utc TIMESTAMP,
    ingestion_lag_seconds DOUBLE, processing_lag_seconds DOUBLE
) USING DELTA""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {ENRICHED_TX_TABLE} (
    event_id STRING, event_type STRING, schema_version STRING,
    transaction_id STRING, source_transaction_id STRING, transaction_id_origin STRING,
    user_id BIGINT, user_id_origin STRING, card_index BIGINT, scoped_card_key STRING,
    amount DECIMAL(18,2), currency STRING, currency_origin STRING,
    merchant_name STRING, merchant_city STRING, merchant_state STRING,
    merchant_latitude DOUBLE, merchant_longitude DOUBLE, zip STRING, mcc INT,
    use_chip STRING, transaction_entry_method STRING, errors STRING,
    source_system STRING, event_time_utc TIMESTAMP, source_event_time_local TIMESTAMP_NTZ,
    event_payload_hash STRING, source_file STRING, source_modified_utc TIMESTAMP,
    kafka_partition INT, kafka_offset BIGINT, kafka_key STRING, ingested_at_utc TIMESTAMP,
    ingestion_lag_seconds DOUBLE, processing_lag_seconds DOUBLE,
    customer_id STRING, user_city STRING, user_region STRING,
    user_fico_score INT, user_age INT, user_num_cards INT, user_total_debt DECIMAL(18,2),
    card_brand STRING, card_type STRING, credit_limit DECIMAL(18,2),
    is_enriched BOOLEAN, enrichment_status STRING
) USING DELTA""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {TRUTH_TABLE} (
    event_id STRING, synthetic_is_fraud BOOLEAN, event_payload_hash STRING,
    scenario STRING, run_id STRING
) USING DELTA""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {QUAR_TABLE} (
    quarantine_id STRING, event_id STRING, quarantine_reason STRING,
    raw_json_line STRING, source_file STRING, kafka_offset BIGINT,
    ingested_at_utc TIMESTAMP, remediation_status STRING
) USING DELTA""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {REG_TABLE} (
    event_id STRING, event_payload_hash STRING, source_file STRING,
    first_ingested_at_utc TIMESTAMP
) USING DELTA""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {PREDS_TABLE} (
    event_id STRING, prediction_score DOUBLE, predicted_label BOOLEAN,
    latency_ms DOUBLE, model_version STRING, predicted_at_utc TIMESTAMP
) USING DELTA""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {DRIFT_TABLE} (
    reconciled_at_utc TIMESTAMP, batch_id BIGINT, total_evaluated BIGINT,
    true_positives BIGINT, false_positives BIGINT, true_negatives BIGINT, false_negatives BIGINT,
    precision DOUBLE, recall DOUBLE, f1_score DOUBLE, mean_latency_ms DOUBLE
) USING DELTA""")

print("ALL DELTA TABLES VERIFIED READY")


## 3. Microbatch Transformation & CDC Enrichment Engine

In [ ]:
def process_silver_microbatch(batch_df, batch_id):
    session = batch_df.sparkSession
    
    def get_attr(k):
        return F.coalesce(
            F.get_json_object(F.col("_biz"), f"$.{k}"),
            F.get_json_object(F.col("_biz"), f"$.['{k}']"),
            F.get_json_object(F.col("_biz"), f"$.\"{k}\"")
        )
    
    parsed = (batch_df
        .withColumn("_val", F.coalesce(F.get_json_object("raw_json_line", "$.value"), F.col("raw_json_line")))
        .withColumn("event_id_p", F.coalesce(F.get_json_object("_val", "$.event_id"), F.col("event_id")))
        .withColumn("event_type_p", F.coalesce(F.get_json_object("_val", "$.event_type"), F.col("event_type")))
        .withColumn("schema_v_p", F.coalesce(F.get_json_object("_val", "$.schema_version"), F.col("schema_version")))
        .withColumn("event_ts_raw", F.coalesce(
            F.get_json_object("_val", "$.event_timestamp"),
            F.get_json_object("_val", "$.event_time"),
            F.get_json_object("_val", "$.timestamp")))
        .withColumn("source_sys", F.get_json_object("_val", "$.source_system"))
        .withColumn("run_id", F.get_json_object("_val", "$.run_id"))
        .withColumn("scenario", F.get_json_object("_val", "$.scenario"))
        .withColumn("_biz", F.coalesce(F.get_json_object("_val", "$.payload"), F.get_json_object("_val", "$.data")))
    )
    
    p = (parsed
        .withColumn("user_raw", F.coalesce(get_attr("User"), get_attr("user_id")))
        .withColumn("card_raw", F.coalesce(get_attr("Card"), get_attr("card_id"), get_attr("card_index")))
        .withColumn("amount_raw", F.coalesce(get_attr("Amount"), get_attr("amount")))
        .withColumn("fraud_raw", F.coalesce(get_attr("Is Fraud?"), get_attr("is_fraud")))
        .withColumn("mcc_raw", F.coalesce(get_attr("MCC"), get_attr("mcc")))
        .withColumn("merchant_name", get_attr("Merchant Name"))
        .withColumn("merchant_city", get_attr("Merchant City"))
        .withColumn("merchant_state", get_attr("Merchant State"))
        .withColumn("merchant_lat_raw", get_attr("Merchant Latitude"))
        .withColumn("merchant_lon_raw", get_attr("Merchant Longitude"))
        .withColumn("zip_raw", get_attr("Zip"))
        .withColumn("use_chip", get_attr("Use Chip"))
        .withColumn("transaction_entry_method", get_attr("Transaction Entry Method"))
        .withColumn("errors", get_attr("Errors?"))
        .withColumn("source_transaction_id", get_attr("transaction_id"))
        .withColumn("user_id", F.expr("try_cast(trim(coalesce(user_raw, kafka_key)) AS BIGINT)"))
        .withColumn("user_id_origin", F.when(F.col("user_raw").isNotNull(), F.lit("PAYLOAD")).otherwise(F.lit("KAFKA_KEY")))
        .withColumn("card_index", F.expr("try_cast(trim(card_raw) AS BIGINT)"))
        .withColumn("mcc", F.expr("try_cast(trim(mcc_raw) AS INT)"))
        .withColumn("merchant_latitude", F.expr("try_cast(trim(merchant_lat_raw) AS DOUBLE)"))
        .withColumn("merchant_longitude", F.expr("try_cast(trim(merchant_lon_raw) AS DOUBLE)"))
        .withColumn("zip", F.regexp_replace(F.trim("zip_raw"), r"\.0$", ""))
        .withColumn("event_time_utc", F.expr("try_to_timestamp(event_ts_raw)"))
        .withColumn("amount_stripped", F.regexp_replace(F.trim("amount_raw"), r"[\$,]", ""))
        .withColumn("amount", F.expr("try_cast(amount_stripped AS DECIMAL(18,2))"))
        .withColumn("currency", F.lit("USD"))
        .withColumn("currency_origin", F.lit("IMPLIED_DOLLAR_SIGN"))
        .withColumn("scoped_card_key", F.concat_ws(":", "user_id", "card_index"))
        .withColumn("transaction_id", F.coalesce(F.col("source_transaction_id"), F.concat(F.lit("EVENT_"), F.col("event_id_p"))))
        .withColumn("transaction_id_origin", F.when(F.col("source_transaction_id").isNotNull(), "SOURCE").otherwise("DERIVED"))
        .withColumn("source_event_time_local", F.col("event_time_utc").cast("timestamp_ntz"))
        .withColumn("kafka_partition", F.lit(0))
        .withColumn("source_system", F.coalesce(F.col("source_sys"), F.lit("POS")))
        .withColumn("synthetic_is_fraud", F.when(F.lower(F.trim("fraud_raw")).isin("yes", "true", "1"), True).otherwise(False))
        .withColumn("event_payload_hash", F.sha2(F.col("raw_json_line"), 256))
        .withColumn("quarantine_id", F.sha2(F.concat_ws("|", "source_file", "kafka_offset", "raw_json_line"), 256))
        .withColumn("ingestion_lag_seconds", F.round(F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.col("event_time_utc")), 2))
        .withColumn("processing_lag_seconds", F.round(F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.col("source_modified_utc")), 2))
    )
    
    reason = (F.when(F.col("event_id_p").isNull() | (F.length("event_id_p") == 0), "MISSING_EVENT_ID")
        .when(F.col("user_id").isNull() | (F.col("user_id") < 0), "MISSING_CARD_IDENTITY")
        .when(F.col("card_index").isNull() | (F.col("card_index") < 0), "MISSING_CARD_IDENTITY")
        .when(F.col("amount").isNull(), "BAD_AMOUNT")
        .when(F.col("event_time_utc").isNull(), "BAD_EVENT_TIME")
        .when((F.col("merchant_latitude").isNotNull() != F.col("merchant_longitude").isNotNull()) |
              (F.abs("merchant_latitude") > 90) | (F.abs("merchant_longitude") > 180), "INVALID_COORDINATES")
        .otherwise(F.lit(None).cast("string")))
    
    evaluated = p.withColumn("quarantine_reason", reason).persist()
    try:
        n = evaluated.count()
        if n == 0:
            return
        good = evaluated.filter(F.col("quarantine_reason").isNull()).dropDuplicates(["event_id_p"])
        bad = evaluated.filter(F.col("quarantine_reason").isNotNull()).dropDuplicates(["quarantine_id"])
        
        # Clean Transactions Surface
        tx = good.select(
            F.col("event_id_p").alias("event_id"), F.col("event_type_p").alias("event_type"), F.col("schema_v_p").alias("schema_version"),
            "transaction_id", "source_transaction_id", "transaction_id_origin", "user_id", "user_id_origin",
            "card_index", "scoped_card_key", "amount", "currency", "currency_origin",
            "merchant_name", "merchant_city", "merchant_state", "merchant_latitude", "merchant_longitude",
            "zip", "mcc", "use_chip", "transaction_entry_method", "errors", "source_system",
            "event_time_utc", "source_event_time_local", "event_payload_hash", "source_file",
            "source_modified_utc", "kafka_partition", "kafka_offset", "kafka_key", "ingested_at_utc",
            "ingestion_lag_seconds", "processing_lag_seconds"
        )
        truth = good.select(F.col("event_id_p").alias("event_id"), "synthetic_is_fraud", "event_payload_hash", "scenario", "run_id")
        quar = bad.select("quarantine_id", F.col("event_id_p").alias("event_id"), "quarantine_reason", "raw_json_line", "source_file", "kafka_offset", "ingested_at_utc").withColumn("remediation_status", F.lit("NEW"))
        reg = good.select(F.col("event_id_p").alias("event_id"), "event_payload_hash", "source_file", F.col("ingested_at_utc").alias("first_ingested_at_utc"))
        
        # CDC Dimension Broadcast Join
        try:
            dim_u = session.table(DIM_USERS).select(
                F.col("source_user_index").alias("dim_uid"), F.col("customer_id").alias("dim_cid"),
                F.col("customer_city").alias("user_city"), F.col("customer_region").alias("user_region"),
                F.col("fico_score").alias("user_fico_score"), F.col("current_age").alias("user_age"),
                F.col("num_credit_cards").alias("user_num_cards"), F.col("total_debt").alias("user_total_debt")
            )
            dim_c = session.table(DIM_CARDS).select(
                F.col("customer_id").alias("dim_c_cid"), F.col("source_card_index").cast("bigint").alias("dim_c_idx"),
                F.col("card_brand"), F.col("card_type"), F.col("credit_limit")
            )
            enriched = (tx.join(F.broadcast(dim_u), tx["user_id"] == dim_u["dim_uid"], "left")
                          .join(F.broadcast(dim_c), (F.col("dim_cid") == F.col("dim_c_cid")) & (tx["card_index"] == F.col("dim_c_idx")), "left")
                          .withColumn("customer_id", F.col("dim_cid"))
                          .withColumn("is_enriched", F.col("user_fico_score").isNotNull())
                          .withColumn("enrichment_status", F.when(F.col("user_fico_score").isNotNull(), "ENRICHED").otherwise("PENDING_CDC"))
                          .drop("dim_uid", "dim_cid", "dim_c_cid", "dim_c_idx"))
        except Exception as e:
            enriched = (tx.withColumn("customer_id", F.lit(None).cast("string"))
                          .withColumn("user_city", F.lit(None).cast("string"))
                          .withColumn("user_region", F.lit(None).cast("string"))
                          .withColumn("user_fico_score", F.lit(None).cast("int"))
                          .withColumn("user_age", F.lit(None).cast("int"))
                          .withColumn("user_num_cards", F.lit(None).cast("int"))
                          .withColumn("user_total_debt", F.lit(None).cast("decimal(18,2)"))
                          .withColumn("card_brand", F.lit(None).cast("string"))
                          .withColumn("card_type", F.lit(None).cast("string"))
                          .withColumn("credit_limit", F.lit(None).cast("decimal(18,2)"))
                          .withColumn("is_enriched", F.lit(False))
                          .withColumn("enrichment_status", F.lit("OFFLINE")))
        
        def merge_insert(tbl, df, v, k):
            df.createOrReplaceTempView(v)
            cols = df.columns
            c_str = ", ".join(f"`{c}`" for c in cols)
            v_str = ", ".join(f"s.`{c}`" for c in cols)
            session.sql(f"MERGE INTO {tbl} AS t USING {v} AS s ON t.`{k}`=s.`{k}` WHEN NOT MATCHED THEN INSERT ({c_str}) VALUES ({v_str})")
        
        merge_insert(TX_TABLE, tx, "_v_tx", "event_id")
        merge_insert(ENRICHED_TX_TABLE, enriched, "_v_enr", "event_id")
        merge_insert(TRUTH_TABLE, truth, "_v_tr", "event_id")
        merge_insert(QUAR_TABLE, quar, "_v_qu", "quarantine_id")
        merge_insert(REG_TABLE, reg, "_v_rg", "event_id")
        
        acc = good.count()
        lag = good.agg(F.avg("ingestion_lag_seconds")).first()[0] or 0.0
        print(f"CONTINUOUS_SILVER_BATCH batch={batch_id} ingested={n} accepted={acc} avg_lag_sec={lag:.2f}")
    finally:
        evaluated.unpersist()


## 4. Launch Concurrent Continuous Streams

In [ ]:
# Stream 1: Bronze AutoLoader
bronze_source = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "text")
    .option("cloudFiles.schemaLocation", RAW_SCHEMA_PATH)
    .option("cloudFiles.useNotifications", "false")
    .option("cloudFiles.useIncrementalListing", "false")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("cloudFiles.allowOverwrites", "false")
    .option("cloudFiles.maxFilesPerTrigger", str(MAX_FILES_PER_TRIGGER))
    .option("pathGlobFilter", "*.json.gz")
    .load(SOURCE_S3))

raw_envelopes = (bronze_source.select(
    F.col("value").alias("raw_json_line"),
    F.col("_metadata.file_path").alias("source_file"),
    F.col("_metadata.file_modification_time").alias("source_modified_utc"),
    F.col("_metadata.file_size").cast("bigint").alias("source_size_bytes"),
    F.get_json_object(F.get_json_object(F.col("value"), "$.value"), "$.event_id").alias("event_id"),
    F.get_json_object(F.get_json_object(F.col("value"), "$.value"), "$.event_type").alias("event_type"),
    F.get_json_object(F.get_json_object(F.col("value"), "$.value"), "$.schema_version").alias("schema_version"),
    F.expr("try_cast(get_json_object(value, '$.offset') AS BIGINT)").alias("kafka_offset"),
    F.get_json_object(F.col("value"), "$.key").alias("kafka_key"),
    F.current_timestamp().alias("ingested_at_utc")
))

q_bronze = (raw_envelopes.writeStream.format("delta")
    .option("checkpointLocation", RAW_CHECKPOINT)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .toTable(RAW_TABLE))
print("STREAM 1 (BRONZE AUTOLOADER) ACTIVE:", q_bronze.id)

# Stream 2: Silver Contract & CDC Enriched Stream
silver_source = spark.readStream.format("delta").table(RAW_TABLE)
q_silver = (silver_source.writeStream.foreachBatch(process_silver_microbatch)
    .option("checkpointLocation", SILVER_CHECKPOINT)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start())
print("STREAM 2 (SILVER CDC ENRICHMENT) ACTIVE:", q_silver.id)

print("BOTH CONTINUOUS STREAMS ARE ACTIVE. Awaiting any termination in 24/7 mode...")
spark.streams.awaitAnyTermination()
